In [0]:
T_trans=spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_item = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")

In [0]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta


offer_start_date = '2025-10-01'
offer_end_date = '2025-10-31'

start = datetime.strptime(offer_start_date, '%Y-%m-%d')

post_window = (start + timedelta(days=31)).strftime('%Y-%m-%d')  # '2025-10-31'
pre_window = (start - timedelta(days=30)).strftime('%Y-%m-%d')   # '2025-09-01'

PY_post_window = (datetime.strptime(post_window, '%Y-%m-%d') - relativedelta(years=1)).strftime('%Y-%m-%d')  # '2024-10-31'
PY_pre_window = (datetime.strptime(pre_window, '%Y-%m-%d') - relativedelta(years=1)).strftime('%Y-%m-%d')   # '2024-09-01'

In [0]:
tags = [
    'BP TAG+$10 - 52 CR',
    'BP TAG+$100 - 802 CR',
    'BP TAG+$129 - 1107 CR',
    'BP TAG+$15 - 72 CR',
    'BP TAG+$25 - 172 CR',
    'BP TAG+$35 - 257 CR',
    'BP TAG+$50 - 377 CR',
    'BP TAG+$75 - 577 CR',
    'PREM TAG+$10 - 54 CR',
    'PREM TAG+$100 - 804 CR',
    'PREM TAG+$129 - 1109 CR',
    'PREM TAG+$15 - 74 CR',
    'PREM TAG+$25 - 174 CR',
    'PREM TAG+$35 - 257 CR',
    'PREM TAG+$35 - 259 CR',
    'PREM TAG+$50 - 379 CR',
    'PREM TAG+$75 - 579 CR',
    'PROMO BP TAG $100 - 1002 CR',
    'TAG $10 - 40 CR',
    'TAG $100 - 790 CR',
    'TAG $129 - 1095 CR',
    'TAG $15 - 60 CR',
    'TAG $25 - 160 CR',
    'TAG $35 - 245 CR',
    'TAG $5 - 20 CR',
    'TAG $50 - 365 CR',
    'TAG $75 - 565 CR',
]

tag_bins = {
    '$5': [
        'TAG $5 - 20 CR',
    ],
    '$10': [
        'BP TAG+$10 - 52 CR',
        'PREM TAG+$10 - 54 CR',
        'TAG $10 - 40 CR',
    ],
    '$15': [
        'BP TAG+$15 - 72 CR',
        'PREM TAG+$15 - 74 CR',
        'TAG $15 - 60 CR',
    ],
    '$25': [
        'BP TAG+$25 - 172 CR',
        'PREM TAG+$25 - 174 CR',
        'TAG $25 - 160 CR',
    ],
    '$35': [
        'BP TAG+$35 - 257 CR',
        'PREM TAG+$35 - 257 CR',
        'PREM TAG+$35 - 259 CR',
        'TAG $35 - 245 CR',
    ],
    '$50': [
        'BP TAG+$50 - 377 CR',
        'PREM TAG+$50 - 379 CR',
        'TAG $50 - 365 CR',
    ],
    '$75': [
        'BP TAG+$75 - 577 CR',
        'PREM TAG+$75 - 579 CR',
        'TAG $75 - 565 CR',
    ],
    '$100': [
        'BP TAG+$100 - 802 CR',
        'PREM TAG+$100 - 804 CR',
        'PROMO BP TAG $100 - 1002 CR',
        'TAG $100 - 790 CR',
    ],
    '$129': [
        'BP TAG+$129 - 1107 CR',
        'PREM TAG+$129 - 1109 CR',
        'TAG $129 - 1095 CR',
    ],
}

In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Row

# Create mapping DataFrame (broadcast - only 27 rows)
mapping_data = [Row(Item_Description=item, price_bin=bin_name) 
                for bin_name, items in tag_bins.items() 
                for item in items]
bin_mapping_df = F.broadcast(spark.createDataFrame(mapping_data))

# Pre-filter dimensions BEFORE joining to fact table
# 1. Date: only keep rows in the needed date ranges
T_date_filtered = (
    T_date
    .select("dateid", "datefull")
    .filter(
        F.col("datefull").between("2024-09-01", "2024-10-31") |
        F.col("datefull").between("2025-09-01", "2025-10-31")
    )
)

# 2. Location: only Restaurant locations
T_location_filtered = (
    T_location
    .select("LocationID", "Province_CD", "LC_Trade_Brand_Description")
    .filter(F.col("LC_Location_Type_Description").like("%Restaurant%"))
)

# 3. Item: only gameband items (small set - broadcast)
T_item_filtered = F.broadcast(
    T_item
    .select("itemskey", "Item_Description")
    .filter(F.col("Item_Description").isin(tags))
)

# Select only needed columns from fact table, then INNER join pre-filtered dimensions
trans_selected = T_trans.select(
    "VisitDateId", "LocationId", "ItemSkey", "Net_Revenue", "Quantity",
    "SCENEFLAG", "CineClubFLAG", "CDE_ID"
)

T_game_band = (
    trans_selected
    .join(T_date_filtered, trans_selected["VisitDateId"] == T_date_filtered["dateid"], "inner")
    .join(T_location_filtered, trans_selected["LocationId"] == T_location_filtered["LocationID"], "inner")
    .join(T_item_filtered, trans_selected["ItemSkey"] == T_item_filtered["itemskey"], "inner")
    .join(bin_mapping_df, on="Item_Description", how="inner")
    .select("datefull", "Province_CD", "LC_Trade_Brand_Description", "Item_Description",
            "price_bin", "Net_Revenue", "Quantity", "SCENEFLAG", "CineClubFLAG", "CDE_ID")
)

In [0]:
import pandas as pd
import numpy as np

# Collect to pandas — result set is small after filtering to 4 months + Restaurants + 27 items
df = T_game_band.toPandas()
df['datefull'] = pd.to_datetime(df['datefull'])


In [0]:
# ── Assign Period labels ──────────────────────────────────────────────────────

PY_offer_start = (datetime.strptime(offer_start_date, '%Y-%m-%d') - relativedelta(years=1)).strftime('%Y-%m-%d')
PY_offer_end = (datetime.strptime(offer_end_date, '%Y-%m-%d') - relativedelta(years=1)).strftime('%Y-%m-%d')

conditions = [
    (df['datefull'].between(offer_start_date, offer_end_date), 'CY_Offer'),
    (df['datefull'].between(pre_window, pd.Timestamp(offer_start_date) - pd.Timedelta(days=1)), 'CY_Pre'),
    (df['datefull'].between(PY_offer_start, PY_offer_end), 'PY_Offer'),
    (df['datefull'].between(PY_pre_window, pd.Timestamp(PY_offer_start) - pd.Timedelta(days=1)), 'PY_Pre'),
]

df['Period'] = np.select([c[0] for c in conditions], [c[1] for c in conditions], default=None)
df_period = df[df['Period'].notna()].copy()

# ── Aggregate by Province, Period, Price Bin ──────────────────────────────────

df_period['Scene_Flag'] = (df_period['SCENEFLAG'] == 'Yes').astype(int)
df_period['CineClub_Flag'] = (df_period['CineClubFLAG'] == 'Yes').astype(int)
df_period['Scene_Revenue_raw'] = df_period['Net_Revenue'].where(df_period['SCENEFLAG'] == 'Yes', 0)

T_sales_mix = (
    df_period
    .groupby(['Province_CD', 'Period', 'price_bin'], as_index=False)
    .agg(
        Total_Sales=('Quantity', 'sum'),
        Total_Revenue=('Net_Revenue', 'sum'),
        Scene_Sales=('Scene_Flag', 'sum'),
        Scene_Revenue=('Scene_Revenue_raw', 'sum'),
        CineClub_Sales=('CineClub_Flag', 'sum'),
        Distinct_Customers=('CDE_ID', 'nunique')
    )
)

# ── Calculate % mix within each Province + Period ─────────────────────────────

T_sales_mix['Sales_Mix_Pct'] = (
    T_sales_mix.groupby(['Province_CD', 'Period'])['Total_Sales']
    .transform(lambda x: (x / x.sum() * 100).round(2))
)
T_sales_mix['Revenue_Mix_Pct'] = (
    T_sales_mix.groupby(['Province_CD', 'Period'])['Total_Revenue']
    .transform(lambda x: (x / x.sum() * 100).round(2))
)
T_sales_mix['Scene_Sales_Mix_Pct'] = (
    T_sales_mix.groupby(['Province_CD', 'Period'])['Scene_Sales']
    .transform(lambda x: (x / x.sum() * 100).round(2))
)
T_sales_mix['Scene_Revenue_Mix_Pct'] = (
    T_sales_mix.groupby(['Province_CD', 'Period'])['Scene_Revenue']
    .transform(lambda x: (x / x.sum() * 100).round(2))
)

T_sales_mix = T_sales_mix.sort_values(['Province_CD', 'Period', 'price_bin']).reset_index(drop=True)

In [0]:
display(T_sales_mix)

In [0]:
import pandas as pd
import numpy as np

# Pull CY_Offer (Oct 1-31 2025) and PY_Offer (Oct 1-31 2024) only — Sep periods excluded
offer_data = T_sales_mix[T_sales_mix['Period'].isin(['CY_Offer', 'PY_Offer'])].copy()

# National rollup per Period + price_bin
nat = (
    offer_data
    .groupby(['price_bin', 'Period'], as_index=False)
    .agg(
        Total_Sales=('Total_Sales', 'sum'),
        Total_Revenue=('Total_Revenue', 'sum'),
        Distinct_Customers=('Distinct_Customers', 'sum'),
        Scene_Sales=('Scene_Sales', 'sum'),
        Scene_Revenue=('Scene_Revenue', 'sum'),
        CineClub_Sales=('CineClub_Sales', 'sum'),
    )
)

# Split CY and PY into separate columns
cy = (nat[nat['Period'] == 'CY_Offer']
      .drop(columns='Period')
      .rename(columns=lambda c: c + '_2025' if c != 'price_bin' else c))
py = (nat[nat['Period'] == 'PY_Offer']
      .drop(columns='Period')
      .rename(columns=lambda c: c + '_2024' if c != 'price_bin' else c))

# Outer merge: keeps new tiers ($129) and new markets (QC) with no 2024 baseline
national = cy.merge(py, on='price_bin', how='outer').fillna(0)

# Cast Decimal columns to float (Spark SQL returns Decimal types)
float_cols = [c for c in national.columns if c != 'price_bin']
national[float_cols] = national[float_cols].astype(float)

# YoY growth for bins that have a 2024 baseline
national['has_2024'] = national['Total_Sales_2024'] > 0

national['YoY_Sales_Rate'] = np.where(
    national['has_2024'],
    (national['Total_Sales_2025'] - national['Total_Sales_2024']) / national['Total_Sales_2024'],
    np.nan
)
national['YoY_Revenue_Rate'] = np.where(
    national['has_2024'],
    (national['Total_Revenue_2025'] - national['Total_Revenue_2024']) / national['Total_Revenue_2024'],
    np.nan
)
national['YoY_Customer_Rate'] = np.where(
    national['has_2024'],
    (national['Distinct_Customers_2025'] - national['Distinct_Customers_2024']) / national['Distinct_Customers_2024'],
    np.nan
)

# Scene YoY rates (bin-level check: some bins may have 0 scene sales in PY)
national['YoY_Scene_Sales_Rate'] = np.where(
    national['Scene_Sales_2024'] > 0,
    (national['Scene_Sales_2025'] - national['Scene_Sales_2024']) / national['Scene_Sales_2024'],
    np.nan
)
national['YoY_Scene_Revenue_Rate'] = np.where(
    national['Scene_Revenue_2024'] > 0,
    (national['Scene_Revenue_2025'] - national['Scene_Revenue_2024']) / national['Scene_Revenue_2024'],
    np.nan
)

# Average YoY applied to new tiers/markets with no 2024 baseline
avg_sales_yoy         = national.loc[national['has_2024'], 'YoY_Sales_Rate'].mean()
avg_revenue_yoy       = national.loc[national['has_2024'], 'YoY_Revenue_Rate'].mean()
avg_customer_yoy      = national.loc[national['has_2024'], 'YoY_Customer_Rate'].mean()
avg_scene_sales_yoy   = national.loc[national['Scene_Sales_2024'] > 0, 'YoY_Scene_Sales_Rate'].mean()
avg_scene_revenue_yoy = national.loc[national['Scene_Revenue_2024'] > 0, 'YoY_Scene_Revenue_Rate'].mean()

national['YoY_Sales_Rate']         = national['YoY_Sales_Rate'].fillna(avg_sales_yoy)
national['YoY_Revenue_Rate']       = national['YoY_Revenue_Rate'].fillna(avg_revenue_yoy)
national['YoY_Customer_Rate']      = national['YoY_Customer_Rate'].fillna(avg_customer_yoy)
national['YoY_Scene_Sales_Rate']   = national['YoY_Scene_Sales_Rate'].fillna(avg_scene_sales_yoy)
national['YoY_Scene_Revenue_Rate'] = national['YoY_Scene_Revenue_Rate'].fillna(avg_scene_revenue_yoy)

# Pct columns for display
national['YoY_Sales_Pct']         = (national['YoY_Sales_Rate']         * 100).round(1)
national['YoY_Revenue_Pct']       = (national['YoY_Revenue_Rate']       * 100).round(1)
national['YoY_Customer_Pct']      = (national['YoY_Customer_Rate']      * 100).round(1)
national['YoY_Scene_Sales_Pct']   = (national['YoY_Scene_Sales_Rate']   * 100).round(1)
national['YoY_Scene_Revenue_Pct'] = (national['YoY_Scene_Revenue_Rate'] * 100).round(1)
national['Note'] = np.where(national['has_2024'], '', '*Avg YoY applied')

# Project 2026 by applying same YoY rates to 2025 actuals
national['Fcst_Sales_2026']     = (national['Total_Sales_2025']        * (1 + national['YoY_Sales_Rate'])).round(0)
national['Fcst_Revenue_2026']   = (national['Total_Revenue_2025']      * (1 + national['YoY_Revenue_Rate'])).round(2)
national['Fcst_Customers_2026'] = (national['Distinct_Customers_2025'] * (1 + national['YoY_Customer_Rate'])).round(0)

national['Fcst_Sales_Chg']    = ((national['Fcst_Sales_2026']     - national['Total_Sales_2025'])        / national['Total_Sales_2025']        * 100).round(1)
national['Fcst_Revenue_Chg']  = ((national['Fcst_Revenue_2026']   - national['Total_Revenue_2025'])      / national['Total_Revenue_2025']      * 100).round(1)
national['Fcst_Customer_Chg'] = ((national['Fcst_Customers_2026'] - national['Distinct_Customers_2025']) / national['Distinct_Customers_2025'] * 100).round(1)

national['Fcst_Scene_Sales_2026']   = (national['Scene_Sales_2025']   * (1 + national['YoY_Scene_Sales_Rate'])).round(0)
national['Fcst_Scene_Revenue_2026'] = (national['Scene_Revenue_2025'] * (1 + national['YoY_Scene_Revenue_Rate'])).round(2)

national['Fcst_Scene_Sales_Chg']   = ((national['Fcst_Scene_Sales_2026']   - national['Scene_Sales_2025'])   / national['Scene_Sales_2025']   * 100).round(1)
national['Fcst_Scene_Revenue_Chg'] = ((national['Fcst_Scene_Revenue_2026'] - national['Scene_Revenue_2025']) / national['Scene_Revenue_2025'] * 100).round(1)

# Sort price_bin in natural order
bin_order = ['$5', '$10', '$15', '$25', '$35', '$50', '$75', '$100', '$129']
national['bin_sort'] = national['price_bin'].map({b: i for i, b in enumerate(bin_order)}).fillna(99)
national = national.sort_values('bin_sort').drop(columns='bin_sort').reset_index(drop=True)

# Totals row
totals = national[[
    'Total_Sales_2024', 'Total_Sales_2025',
    'Total_Revenue_2024', 'Total_Revenue_2025',
    'Distinct_Customers_2024', 'Distinct_Customers_2025',
    'Scene_Sales_2024', 'Scene_Revenue_2024',
    'Scene_Sales_2025', 'Scene_Revenue_2025',
    'Fcst_Sales_2026', 'Fcst_Revenue_2026', 'Fcst_Customers_2026',
    'Fcst_Scene_Sales_2026', 'Fcst_Scene_Revenue_2026',
]].sum()
totals['price_bin'] = 'TOTAL'
totals['YoY_Sales_Pct']    = round((totals['Total_Sales_2025'] - totals['Total_Sales_2024']) / totals['Total_Sales_2024'] * 100, 1)
totals['YoY_Revenue_Pct']  = round((totals['Total_Revenue_2025'] - totals['Total_Revenue_2024']) / totals['Total_Revenue_2024'] * 100, 1)
totals['YoY_Customer_Pct'] = round((totals['Distinct_Customers_2025'] - totals['Distinct_Customers_2024']) / totals['Distinct_Customers_2024'] * 100, 1)
totals['Fcst_Sales_Chg']    = round((totals['Fcst_Sales_2026'] - totals['Total_Sales_2025']) / totals['Total_Sales_2025'] * 100, 1)
totals['Fcst_Revenue_Chg']  = round((totals['Fcst_Revenue_2026'] - totals['Total_Revenue_2025']) / totals['Total_Revenue_2025'] * 100, 1)
totals['Fcst_Customer_Chg']      = round((totals['Fcst_Customers_2026']    - totals['Distinct_Customers_2025']) / totals['Distinct_Customers_2025'] * 100, 1)
totals['Fcst_Scene_Sales_Chg']   = round((totals['Fcst_Scene_Sales_2026']   - totals['Scene_Sales_2025'])   / totals['Scene_Sales_2025']   * 100, 1)
totals['Fcst_Scene_Revenue_Chg'] = round((totals['Fcst_Scene_Revenue_2026'] - totals['Scene_Revenue_2025']) / totals['Scene_Revenue_2025'] * 100, 1)
totals['YoY_Scene_Sales_Pct']   = round((totals['Scene_Sales_2025']   - totals['Scene_Sales_2024'])   / totals['Scene_Sales_2024']   * 100, 1)
totals['YoY_Scene_Revenue_Pct'] = round((totals['Scene_Revenue_2025'] - totals['Scene_Revenue_2024']) / totals['Scene_Revenue_2024'] * 100, 1)
totals['Note'] = ''
national = pd.concat([national, pd.DataFrame([totals])], ignore_index=True)

# Table 1: Oct 2024 Actuals
print('=== Oct 2024 Actuals  [Period: 2024-10-01 to 2024-10-31] ===')
display(national[['price_bin', 'Total_Sales_2024', 'Total_Revenue_2024',
                   'Distinct_Customers_2024', 'Scene_Sales_2024', 'Scene_Revenue_2024']]
        .rename(columns={
            'Total_Sales_2024': 'Total_Sales', 'Total_Revenue_2024': 'Total_Revenue',
            'Distinct_Customers_2024': 'Distinct_Customers',
            'Scene_Sales_2024': 'Scene_Sales', 'Scene_Revenue_2024': 'Scene_Revenue',
        }))

# Table 2: Oct 2025 Actuals
print('=== Oct 2025 Actuals  [Period: 2025-10-01 to 2025-10-31]  |  YoY vs Oct 2024 ===')
display(national[['price_bin', 'Total_Sales_2025', 'Total_Revenue_2025',
                   'Distinct_Customers_2025', 'Scene_Sales_2025', 'Scene_Revenue_2025',
                   'YoY_Sales_Pct', 'YoY_Revenue_Pct', 'YoY_Customer_Pct',
                   'YoY_Scene_Sales_Pct', 'YoY_Scene_Revenue_Pct']]
        .rename(columns={
            'Total_Sales_2025': 'Total_Sales', 'Total_Revenue_2025': 'Total_Revenue',
            'Distinct_Customers_2025': 'Distinct_Customers',
            'Scene_Sales_2025': 'Scene_Sales', 'Scene_Revenue_2025': 'Scene_Revenue',
            'YoY_Sales_Pct': 'YoY_Sales_%', 'YoY_Revenue_Pct': 'YoY_Revenue_%',
            'YoY_Customer_Pct': 'YoY_Customer_%',
            'YoY_Scene_Sales_Pct': 'YoY_Scene_Sales_%', 'YoY_Scene_Revenue_Pct': 'YoY_Scene_Revenue_%',
        }))

# Table 3: Oct 2026 Forecast
print(f'=== Oct 2026 Forecast  [Period: 2026-10-01 to 2026-10-31]  |  Method: YoY | Avg fallback = {avg_sales_yoy*100:.1f}% units / {avg_revenue_yoy*100:.1f}% revenue ===')
display(national[['price_bin',
                   'Fcst_Sales_2026', 'Fcst_Sales_Chg',
                   'Fcst_Revenue_2026', 'Fcst_Revenue_Chg',
                   'Fcst_Customers_2026', 'Fcst_Customer_Chg',
                   'Fcst_Scene_Sales_2026', 'Fcst_Scene_Sales_Chg',
                   'Fcst_Scene_Revenue_2026', 'Fcst_Scene_Revenue_Chg',
                   'Note']]
        .rename(columns={
            'Fcst_Sales_2026': 'Fcst_Sales', 'Fcst_Sales_Chg': 'Sales_Chg_%',
            'Fcst_Revenue_2026': 'Fcst_Revenue', 'Fcst_Revenue_Chg': 'Revenue_Chg_%',
            'Fcst_Customers_2026': 'Fcst_Customers', 'Fcst_Customer_Chg': 'Customer_Chg_%',
            'Fcst_Scene_Sales_2026': 'Fcst_Scene_Sales', 'Fcst_Scene_Sales_Chg': 'Scene_Sales_Chg_%',
            'Fcst_Scene_Revenue_2026': 'Fcst_Scene_Revenue', 'Fcst_Scene_Revenue_Chg': 'Scene_Revenue_Chg_%',
        }))

# October 2026 Gameband Forecast — Methodology

## Forecast Period
**October 1–31, 2026** (national, all provinces)

---

## Method: Year-over-Year (YoY) Projection

The forecast applies the observed October 2024 → October 2025 growth rate for each price bin to the October 2025 actuals.

```
Forecast Oct-26 = Actual Oct-25 × (1 + YoY Rate)
where YoY Rate = (Oct-25 Actual − Oct-24 Actual) / Oct-24 Actual
```

This approach is chosen because gamebands are a **promotional, seasonal product** — the October offer window is the relevant comparable period, and same-month YoY comparison removes seasonal noise automatically.

---

## Data Source
- **Source table:** `cpx_dataanalytics_gold.customer360.c360_f_transaction` joined to item, date, and location dimensions
- **CY_Offer:** Oct 1–31, 2025 (current year offer window)
- **PY_Offer:** Oct 1–31, 2024 (prior year offer window)
- **Scope:** Restaurant locations, Scene Gameband SKUs only (27 items across 8 price tiers)
- **Geography:** National rollup (AB, BC, MB, NL, NS, ON, QC)

---

## Fallback Rule — New Bins / New Markets
Price bins or provinces with **no October 2024 baseline** ($129 tier, QC market) use the **national average YoY rate** across all comparable bins as a proxy. These rows are flagged with `*Avg YoY applied` in the output.

---

## Metrics Forecasted
| Metric | Method |
|---|---|
| Total Sales (units) | YoY rate on Oct-25 units |
| Total Revenue | YoY rate on Oct-25 revenue |
| Distinct Customers | YoY rate on Oct-25 customers |
| Scene Sales (units) | Scene-specific YoY rate on Oct-25 Scene units |
| Scene Revenue | Scene-specific YoY rate on Oct-25 Scene revenue |

---

## Final Forecast — October 2026

| Price Bin | Oct-25 Actual | YoY Rate | **Fcst Oct-26 Units** | **Fcst Oct-26 Revenue** |
|---|---|---|---|---|
| $10 | 5,554 | +13.6% | **6,311** | **$72,481** |
| $25 | 18,234 | +6.0% | **19,322** | **$522,294** |
| $35 | 15,436 | +15.0% | **17,755** | **$662,768** |
| $50 | 16,382 | +36.2% | **22,308** | **$1,175,934** |
| $75 | 4,983 | +62.1% | **8,078** | **$621,629** |
| $100 | 5,267 | +2.3% | **5,388** | **$550,292** |
| $129 | 830 | +19.3%* | **990** | **$125,195** |
| **TOTAL** | **66,689** | **+20.2%** | **80,155** | **$3,730,609** |

*\*Avg YoY fallback applied — no Oct 2024 baseline for this tier*

---

## Key Assumptions & Caveats
1. **Promotional parity** — Forecast assumes October 2026 carries a comparable promotional offer to October 2025. If the promotion is weaker or absent, actual sales may be materially lower.
2. **No external factors** — Pricing changes, competitive activity, and macroeconomic shifts are not modelled.
3. **$75 tier caution** — The +62.1% YoY rate reflects a strong 2024→2025 launch effect. This rate is unlikely to sustain at the same magnitude.
4. **Conservative downside scenario** — Applying the recent non-October monthly YoY trend (~−15%) to Oct-25 actuals gives a downside of approximately **~56,700 units**.
5. **QC market** — Included in Oct-25 actuals but absent from Oct-24 comparables; national YoY is slightly inflated as a result.